## Optuna HBV calibration test

This notebook combines the HBV setup from the CMA-ES workflow with Optuna-based calibration.

In [ ]:
# setup imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path().resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import warnings

import seaborn as sns
from dask import delayed
from dask.distributed import Client, progress
from ewatercycle.observation.grdc import get_grdc_data

from src.forcing import generate_lumped_ERA5_forcing, load_lumped_forcing_data
from src.models import *
from src.paths import *

warnings.filterwarnings("ignore", category=UserWarning)

# Keep imports available for later cells without linter "unused import" warnings.
_ = (plt, np, pd, sns, delayed, Client, progress)

In [ ]:
shape_name = "Chatly_GRDC"
start_date = "1955-01-02T00:00Z"
end_date = "1959-12-31T00:00Z"

generate_lumped_ERA5_forcing(shape_name, start=start_date, end=end_date)
ERA5_forcing_loaded = load_lumped_forcing_data("Chatly_GRDC", "ERA5", "1955-1959")

grdc_station = get_grdc_data(
    2817100, "1955-01-02T00:00Z", "1959-12-31T00:00Z", data_home=GRDC / "Daily"
)

q_obs = grdc_station["streamflow"]

In [ ]:
# Optuna setup
x0_norm = scale(0.5 * (p_min + p_max))

seeds = [
    180988,
    214025,
    987654,
    123456,
    654321,
    111222,
    333444,
    555666,
    777888,
    999000,
    112233,
    445566,
    778899,
    101010,
    202020,
    303030,
    404040,
    505050,
    606060,
    707070,
]

In [ ]:
# Create a Dask client with 3 workers, 1 thread each
dask_client = Client(n_workers=3, threads_per_worker=1, dashboard_address=":0")
dask_client

In [ ]:
objective_fn = make_objective_safe(ERA5_forcing_loaded, q_obs.values, shape_name)
number_runs = 12
test_seeds = seeds[:number_runs]
n_trials = 240

tasks = [
    delayed(run_optuna)(
        optuna_seed=s,
        objective_fn=objective_fn,
        n_trials=n_trials,
        n_jobs=1,
        show_progress_bar=False,
        save_folder="results_optuna_chatly/",
    )
    for s in test_seeds
]

futures = dask_client.compute(tasks)
progress(futures)
results = dask_client.gather(futures)

In [ ]:
# Summary of each Optuna run
df_results = pd.DataFrame(results)
df_results = df_results[["seed", "best_f", "best_x", "best_x_phys", "n_trials", "study_name"]]
df_results.sort_values("best_f").reset_index(drop=True)

In [ ]:
# Best parameter row per seed based on objective history
best_params_list = []

for run in results:
    seed = run["seed"]
    history = run["history"]

    best_idx = int(np.argmin(history["objective"]))
    best_theta_phys = history["theta_phys"][best_idx]

    row = {
        "seed": seed,
        "best_eval": best_idx + 1,
        "objective": history["objective"][best_idx],
        "nse": history["nse"][best_idx],
        "kge": history["kge"][best_idx],
        "vol_err": history["vol_err"][best_idx],
    }

    for i, th in enumerate(best_theta_phys):
        row[parameter_names[i]] = th

    best_params_list.append(row)

best_params_df = pd.DataFrame(best_params_list)
best_params_df.sort_values("objective").reset_index(drop=True)

In [ ]:
# Visualize spread of scaled best parameters
param_cols = ["Imax", "Ce", "Sumax", "Beta", "Pmax", "Tlag", "Kf", "Ks", "FM"]
scaled_params_df = pd.DataFrame(scale(best_params_df[param_cols].to_numpy()), columns=param_cols)

plt.figure(figsize=(12, 4))
sns.boxplot(data=scaled_params_df, color="lightgray", fliersize=0)
sns.stripplot(data=scaled_params_df, color="black", jitter=True, size=6, marker="o")
plt.xticks(rotation=45)
plt.ylabel("Scaled parameter value (0-1)")
plt.title("Scaled best Optuna parameter sets across seeds")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# Cleanly close the local Dask cluster when finished
dask_client.close()